In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install torchmetrics

In [3]:
import random
import numpy as np
import torch

def set_seed(seed):
  random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)

In [11]:
# config.py

from pathlib import Path
import re as _re

def get_config():
  return {
      "datasource": "wmt/wmt14",
      "dataset_config": "fr-en",
      "lang_src": "en",
      "lang_tgt": "fr",
      "model_folder": "weights",
      "model_basename": "tmodel_",
      "tokenizer_file": "tokenizer_{0}.json",
      "shuffle_seed": 0,
      "vocab_size": 32000,
      "max_seq": 300,
      "num_workers": 8,
      "prefetch_factor": 4,
      "batch_size": 256,
      "beam_size": 4,
      "length_penalty": 0.6,
      "num_validation_examples": 3,
      "validation_size": 3000,
      "d_model": 512,
      "h": 8,
      "N": 6,
      "dropout": 0.0,
      "betas": (0.9, 0.98),
      "eps": 1e-9,
      "warmup_steps": 1500,
      "lr_scale": 1.0,
      "preload": "latest",
      "use_compile": True,
      "label_smoothing": 0.1,
      "num_epochs": 1,
      "decode_strategy": "beam",
      "num_pairs": 4_040_000,
      "val_loss_every_pairs": 100_000,
      "bleu_every_pairs": 5_000_000,
      "cache_dir": "/content/drive/MyDrive/mt_cache",
      "seed": 0,
  }

def weights_folder(config):
  return f"{config['datasource'].replace('/', '_')}_{config['model_folder']}"

def get_weights_file_path(config, epoch: str):
  return str(Path(".") / weights_folder(config) / f"{config['model_basename']}{epoch}.pt")

def latest_weights_file_path(config):
  files = sorted(Path(weights_folder(config)).glob(f"{config['model_basename']}*"))
  return str(files[-1]) if files else None

def tokenizer_path(config, name: str) -> Path:
  slug = config["datasource"].split("/")[-1]
  cfg_slug = config["dataset_config"]
  d = Path(config.get("cache_dir", "."))
  d.mkdir(parents=True, exist_ok=True)
  return d / config["tokenizer_file"].format(f"{slug}_{cfg_slug}_s{config['shuffle_seed']}_{name}")

def build_tokenizer(vocab_size, special_tokens):
    from tokenizers import Tokenizer, decoders, pre_tokenizers
    from tokenizers.models import WordPiece
    from tokenizers.trainers import WordPieceTrainer

    tok = Tokenizer(WordPiece(unk_token="[UNK]"))
    tok.pre_tokenizer = pre_tokenizers.Sequence([
        pre_tokenizers.WhitespaceSplit(),
        pre_tokenizers.Split(pattern="'", behavior="merged_with_previous"),
    ])
    tok.decoder = decoders.WordPiece(prefix="##", cleanup=True)
    trainer = WordPieceTrainer(vocab_size=vocab_size, min_frequency=2,
                               special_tokens=special_tokens, show_progress=False)
    return tok, trainer


_APOS = _re.compile(r"\s*'\s*")
_HYPH = _re.compile(r"\s+-\s+")
_PUNC = _re.compile(r"\s*([.,;:!?])")
_MULTI = _re.compile(r"\s+")

def clean_output(text: str) -> str:
  text = _APOS.sub("'", text)
  text = _HYPH.sub("-", text)
  text = _PUNC.sub(r"\1", text)
  return _MULTI.sub(" ", text).strip()

In [12]:
# dataset.py

import random
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, Sampler

class TranslationDataset(Dataset):
  def __init__(self, tokenizer, raw_ds, src_lng, tgt_lng, max_seq):
    super().__init__()
    self.pad_id, self.sos_id, self.eos_id = tokenizer.token_to_id('[PAD]'), tokenizer.token_to_id('[SOS]'), tokenizer.token_to_id('[EOS]')
    self.samples, self.lengths = [], []
    self.dropped_long = self.dropped_junk = 0
    for item in raw_ds:
      src_text = item['translation'][src_lng].strip()
      tgt_text = item['translation'][tgt_lng].strip()

      if not src_text or not tgt_text:
        self.dropped_junk += 1; continue

      if src_text == tgt_text:
        self.dropped_junk += 1; continue

      src_ids = tokenizer.encode(src_text).ids
      tgt_ids = tokenizer.encode(tgt_text).ids
      max_length = max(len(src_ids) + 2, len(tgt_ids) + 1)
      if max_length >= max_seq :
        self.dropped_long += 1; continue

      if len(src_ids) < 1 or len(tgt_ids) < 1:
        self.dropped_junk += 1; continue

      r = max(len(src_ids), len(tgt_ids)) / min(len(src_ids), len(tgt_ids))
      if r > 2.5:
        self.dropped_junk += 1; continue


      self.samples.append((src_text, tgt_text, src_ids, tgt_ids))
      self.lengths.append(max(len(src_ids) + 2, len(tgt_ids) + 1))
    print(f"junk_items_dropped: {self.dropped_junk} ; long_items_dropped: {self.dropped_long}")

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx):
    src_text, tgt_text, src_ids, tgt_ids = self.samples[idx]
    return {
        "enc_input": torch.tensor([self.sos_id, *src_ids, self.eos_id], dtype = torch.long),
        "dec_input": torch.tensor([self.sos_id, *tgt_ids], dtype = torch.long),
        "label": torch.tensor([*tgt_ids, self.eos_id], dtype = torch.long),
        "src_txt": src_text, "tgt_txt": tgt_text
    }

def make_collate_fn(pad_id):
  def collate_fn(batch):
    enc = pad_sequence([b["enc_input"] for b in batch], batch_first = True, padding_value = pad_id)
    dec = pad_sequence([b["dec_input"] for b in batch], batch_first = True, padding_value = pad_id)
    lbl = pad_sequence([b["label"] for b in batch], batch_first = True, padding_value = pad_id)
    enc_mask = (enc != pad_id).unsqueeze(1).unsqueeze(1)

    return {"enc_input": enc, "dec_input": dec, "label": lbl,
            "enc_mask": enc_mask,
            "src_txt": [b["src_txt"] for b in batch],
            "tgt_txt": [b["tgt_txt"] for b in batch]
            }

  return collate_fn

class LengthBatchSampler(Sampler):
  def __init__(self, lengths, batch_size, shuffle = True, mega_factor = 50):
    self.lengths = lengths
    self.batch_size = batch_size
    self.shuffle = shuffle
    self.mega = batch_size * mega_factor

  def __len__(self):
    return (len(self.lengths) + self.batch_size - 1) // self.batch_size

  def __iter__(self):
    idx = list(range(len(self.lengths)))
    if self.shuffle:
      random.shuffle(idx)
    batches = []
    for i in range(0, len(idx), self.mega):
      chunk = sorted(idx[i:i + self.mega], key=lambda j: self.lengths[j])
      batches += [chunk[k:k + self.batch_size] for k in range(0, len(chunk), self.batch_size)]
    if self.shuffle:
      random.shuffle(batches)
    yield from batches


def _ds_from_blob(tok, samples, lengths):
  ds = TranslationDataset.__new__(TranslationDataset)
  ds.pad_id = tok.token_to_id('[PAD]')
  ds.sos_id = tok.token_to_id('[SOS]')
  ds.eos_id = tok.token_to_id('[EOS]')
  ds.samples, ds.lengths = samples, lengths
  ds.dropped_long = ds.dropped_junk = 0
  return ds

In [21]:
# model.py

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiHeadAttention(nn.Module):
  def __init__(self, d_model: int, h: int, dropout: float):
    super().__init__()
    self.w_q = nn.Linear(d_model, d_model, bias = False)
    self.w_k = nn.Linear(d_model, d_model, bias = False)
    self.w_v = nn.Linear(d_model, d_model, bias = False)
    self.w_o = nn.Linear(d_model, d_model, bias = False)
    self.d_model = d_model
    self.h = h
    assert d_model % h == 0, "d_model should be divisble by the number of heads"
    self.d_k = d_model // h
    self.dropout = dropout
    self.scale = self.d_k ** -0.5

  def split_heads(self, x):
    return x.reshape(x.shape[0], x.shape[1], self.h, self.d_k).swapaxes(1,2)

  def forward(self, q, k, v, mask = None, is_causal: bool = False, is_fast: bool = True):
    # mask is a boolean Tensor where True is a real token
    # (B, seq_len, d_model)
    q, k, v = self.split_heads(self.w_q(q)), self.split_heads(self.w_k(k)), self.split_heads(self.w_v(v)) # (B, h, seq_len, d_k)

    if is_fast:
      x = F.scaled_dot_product_attention(q, k, v, mask, self.dropout if self.training else 0.0, is_causal)
    else:
      scores = (q @ k.swapaxes(-1, -2)) * self.scale
      if is_causal:
        causal = torch.ones((scores.shape[-2], scores.shape[-1]), device=scores.device, dtype=torch.bool).tril()
        mask = causal if mask is None else (mask & causal)
      if mask is not None:
        scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
      probs = F.dropout(scores.softmax(-1), self.dropout, self.training)
      x = probs @ v

    # x.shape == (B, h, seq_len, d_k)
    x = x.swapaxes(1,2).reshape(x.shape[0], x.shape[2], self.d_model)
    return self.w_o(x) # (B, seq_len, d_model)

class EncoderBlock(nn.Module):
  def __init__(self, d_model, h, dropout):
    super().__init__()
    self.self_mha = MultiHeadAttention(d_model, h, dropout)
    self.ln1 = nn.LayerNorm(d_model)
    self.ffn = nn.Sequential(nn.Linear(d_model, d_model * 4), nn.ReLU(), nn.Linear(d_model * 4, d_model))
    self.ln2 = nn.LayerNorm(d_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x, mask):
    # mask shape should be (B, 1, 1, S)
    h = self.ln1(x)
    x = self.dropout(self.self_mha(h, h, h, mask)) + x
    return self.dropout(self.ffn(self.ln2(x))) + x

class DecoderBlock(nn.Module):
  def __init__(self, d_model, h, dropout):
    super().__init__()
    self.self_causal_mha = MultiHeadAttention(d_model, h, dropout)
    self.ln1 = nn.LayerNorm(d_model)
    self.cross_mha = MultiHeadAttention(d_model, h, dropout)
    self.ln2 = nn.LayerNorm(d_model)
    self.ffn = nn.Sequential(nn.Linear(d_model, d_model * 4), nn.ReLU(), nn.Linear(d_model * 4, d_model))
    self.ln3 = nn.LayerNorm(d_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self, tgt, enc_out, src_mask):
    h = self.ln1(tgt)
    x = self.dropout(self.self_causal_mha(h, h, h, None, True)) + tgt
    x = self.dropout(self.cross_mha(self.ln2(x), enc_out, enc_out, src_mask)) + x
    return self.dropout(self.ffn(self.ln3(x))) + x


def pe(d_model, max_seq):
  pos = torch.arange(0, max_seq).unsqueeze(1)
  i2 = torch.arange(0, d_model, 2)
  ang = pos * torch.exp(-i2 / d_model * math.log(10000.0))
  out = torch.zeros(max_seq, d_model)
  out[:, 0::2] = torch.sin(ang)
  out[:, 1::2] = torch.cos(ang)

  return out


class Transformer(nn.Module):
  def __init__(self, d_model, h, N, vocab_sz, max_seq, dropout):
    super().__init__()
    self.dropout = nn.Dropout(dropout)
    self.max_seq = max_seq
    self.d_model = d_model
    self.embed = nn.Embedding(vocab_sz, d_model)
    self.encoder = nn.ModuleList([EncoderBlock(d_model, h, dropout) for _ in range(N)])
    self.decoder = nn.ModuleList([DecoderBlock(d_model, h, dropout) for _ in range(N)])
    self.enc_final_ln = nn.LayerNorm(d_model)
    self.dec_final_ln = nn.LayerNorm(d_model)
    self.lm_head = nn.Linear(d_model, vocab_sz, False)
    self.lm_head.weight = self.embed.weight
    self._init_weights()
    self.register_buffer('pe', pe(d_model, max_seq))

  def _init_weights(self):
    for p in self.parameters():
      if p.dim() > 1:
        nn.init.xavier_uniform_(p)
    nn.init.normal_(self.embed.weight, mean=0.0, std=self.d_model ** -0.5)


  def _embed(self, ids):
    x = self.embed(ids) * math.sqrt(self.d_model)
    x = x + self.pe[:ids.shape[-1], :]
    return self.dropout(x)

  def encode(self, src, src_mask):
    x = self._embed(src)
    for encoder_block in self.encoder:
      x = encoder_block(x, src_mask)
    return self.enc_final_ln(x)

  def decode(self, memory, src_mask, tgt):
    x = self._embed(tgt)
    for decoder_block in self.decoder:
      x = decoder_block(x, memory, src_mask)
    return self.dec_final_ln(x)

  def proj(self, x):
    return self.lm_head(x)

  def forward(self, src, src_mask, tgt):
    assert tgt.shape[-1] <= self.max_seq and src.shape[-1] <= self.max_seq
    return self.proj(self.decode(self.encode(src, src_mask), src_mask, tgt))

@torch.no_grad()
def greedy_decode(model, src, src_mask, sos, eos, pad, max_len):
  B, device = src.size(0), src.device
  memory = model.encode(src, src_mask)
  ys = torch.full((B, 1), sos, dtype=torch.long, device=device)
  finished = torch.zeros(B, dtype=torch.bool, device=device)
  for _ in range(max_len - 1):
    logits = model.proj(model.decode(memory, src_mask, ys)[:, -1])
    nxt = logits.argmax(-1).masked_fill(finished, pad)
    ys = torch.cat([ys, nxt.unsqueeze(1)], dim=1)
    finished |= nxt == eos
    if bool(finished.all()):
      break
  return ys


@torch.no_grad()
def beam_search_decode(model, src, src_mask, sos, eos, max_len, beam_size=4, alpha=0.6):
  device = src.device
  memory = model.encode(src, src_mask)
  seqs = torch.full((1, 1), sos, dtype=torch.long, device=device)
  scores = torch.zeros(1, device=device)
  finished = []

  for _ in range(max_len - 1):
    n, vocab = seqs.size(0), model.lm_head.out_features
    logits = model.proj(model.decode(memory.expand(n, -1, -1),
                                      src_mask.expand(n, -1, -1, -1), seqs)[:, -1])
    logp = torch.log_softmax(logits, dim=-1)

    cand = (scores.unsqueeze(1) + logp).reshape(-1)
    scores, flat = cand.topk(min(beam_size, cand.numel()))
    seqs = torch.cat([seqs[flat // vocab], (flat % vocab).unsqueeze(1)], dim=1)

    done = seqs[:, -1] == eos
    for j in done.nonzero().flatten().tolist():
      finished.append((seqs[j], scores[j].item()))
    seqs, scores = seqs[~done], scores[~done]
    if seqs.size(0) == 0 or len(finished) >= beam_size:
      break

  finished += [(seqs[j], scores[j].item()) for j in range(seqs.size(0))]
  lp = lambda L: ((5 + L) / 6) ** alpha
  return max(finished, key=lambda t: t[1] / lp(t[0].size(0)))[0]



In [22]:
# train.py

import random
import numpy as np
import torch

from tqdm import tqdm
from datasets import load_dataset
from itertools import islice
from tokenizers import Tokenizer
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import LambdaLR
import torchmetrics
import warnings
import math

import pickle, hashlib, gc, time
from pathlib import Path

_DS_MEMO = {}

def _ds_key(cfg):
  return "|".join(str(cfg[k]) for k in
    ["datasource","dataset_config","lang_src","lang_tgt",
     "shuffle_seed","num_pairs","vocab_size","max_seq"])

def _ds_cache_file(cfg):
  h = hashlib.md5(_ds_key(cfg).encode()).hexdigest()[:12]
  d = Path(cfg.get("cache_dir", "."))
  d.mkdir(parents=True, exist_ok=True)
  return d / f"ds_{h}.pkl"

def load_pairs(cfg):
  stream = load_dataset(cfg["datasource"], cfg["dataset_config"], split = "train", streaming = True)
  stream = stream.shuffle(seed = cfg["shuffle_seed"], buffer_size = 50000)
  train_rows = list(tqdm(islice(stream, cfg["num_pairs"]),
                           total=cfg["num_pairs"], desc="loading train", unit="pair"))
  val_rows = None
  try:
    val_stream = list(load_dataset(cfg["datasource"], cfg["dataset_config"], split="validation", streaming =  True))
    val_rows = list(islice(val_stream, cfg.get("validation_size", 1000)))
  except Exception as e:
    print(f"No official validation split, {e}, will create one from train split")
  if val_rows is None:
    k = len(train_rows) // 100
    val_rows, train_rows = train_rows[:k], train_rows[k:]
  return train_rows, val_rows


def get_or_build_tokenizer(cfg, rows, langs: list[str], name):
  path = tokenizer_path(cfg, name)
  if path.exists():
    return Tokenizer.from_file(str(path))
  tok, trainer = build_tokenizer(cfg["vocab_size"], ["[UNK]", "[PAD]", "[SOS]", "[EOS]"])
  tok.train_from_iterator((row["translation"][l] for row in rows for l in langs), trainer)
  tok.save(str(path))
  return tok


def get_ds(cfg):
  src, tgt = cfg["lang_src"], cfg["lang_tgt"]
  key, path, tok_path = _ds_key(cfg), _ds_cache_file(cfg), tokenizer_path(cfg, "shared")


  if key in _DS_MEMO:
    train_ds, val_ds, tok = _DS_MEMO[key]
    print("dataset: memory cache hit")

  elif path.exists() and tok_path.exists():
    print(f"dataset: loading {path}")
    t0 = time.time()
    with open(path, "rb") as f:
      blob = pickle.load(f)
    tok = Tokenizer.from_file(str(tok_path))
    train_ds = _ds_from_blob(tok, *blob["train"])
    val_ds   = _ds_from_blob(tok, *blob["val"])
    _DS_MEMO[key] = (train_ds, val_ds, tok)
    print(f"dataset: loaded in {time.time()-t0:.0f}s")

  else:
    print("dataset: cold build (~15 min, happens once)")
    train_rows, val_rows = load_pairs(cfg)
    tok = get_or_build_tokenizer(cfg, train_rows, [src, tgt], "shared")
    train_ds = TranslationDataset(tok, train_rows, src, tgt, cfg["max_seq"])
    val_ds   = TranslationDataset(tok, val_rows,   src, tgt, cfg["max_seq"])
    del train_rows, val_rows; gc.collect()

    if cfg.get("strip_train_text", True):
      train_ds.samples = [("", "", s, t) for _, _, s, t in train_ds.samples]

    tmp = path.with_suffix(".tmp")
    with open(tmp, "wb") as f:
      pickle.dump({"train": (train_ds.samples, train_ds.lengths),
                   "val":   (val_ds.samples,   val_ds.lengths)}, f, protocol=5)
    tmp.rename(path)
    _DS_MEMO[key] = (train_ds, val_ds, tok)
    print(f"dataset: cached to {path}")

  print(f"Vocab {tok.get_vocab_size()} | train {len(train_ds)} val {len(val_ds)}")
  args = dict(collate_fn=make_collate_fn(tok.token_to_id("[PAD]")),
              num_workers=cfg['num_workers'], pin_memory=True,
              persistent_workers=True, prefetch_factor=cfg["prefetch_factor"])
  train_dl = DataLoader(train_ds, batch_sampler=LengthBatchSampler(train_ds.lengths, cfg["batch_size"]), **args)
  val_dl   = DataLoader(val_ds,   batch_sampler=LengthBatchSampler(val_ds.lengths, cfg["batch_size"], shuffle=False), **args)
  return train_dl, val_dl, tok


def ids_to_text(row, tok, sos, eos):
  ids = row.tolist()
  ids = ids[1:] if ids[0] == sos else ids
  if eos in ids:
    ids = ids[:ids.index(eos)]
  return clean_output(tok.decode(ids))

@torch.no_grad()
def run_validation_loss(model, val_dl, eval_loss_fn, pad, vocab, device):
  model.eval()
  total_loss, total_tokens = 0.0, 0
  for batch in val_dl:
    enc = batch["enc_input"].to(device, non_blocking=True)
    dec = batch["dec_input"].to(device, non_blocking=True)
    enc_mask = batch["enc_mask"].to(device, non_blocking=True)
    label = batch["label"].to(device, non_blocking=True)

    with torch.autocast("cuda", dtype=torch.bfloat16):
      logits = model(enc, enc_mask, dec)
      loss = eval_loss_fn(logits.view(-1, vocab), label.view(-1))

    total_loss += loss.item()
    total_tokens += (label != pad).sum().item()

  return total_loss / max(total_tokens, 1)

@torch.no_grad()
def run_validation(model, val_dl, tok, cfg, device):
  model.eval()
  sos, eos, pad = (tok.token_to_id(t) for t in ("[SOS]", "[EOS]", "[PAD]"))
  expected = []
  predicted = []
  sources = []

  for batch in val_dl:
    enc = batch["enc_input"].to(device)
    enc_mask = batch["enc_mask"].to(device)

    rows = greedy_decode(model, enc, enc_mask, sos, eos, pad, cfg["max_seq"])
    for row, src_t, tgt_t in zip(rows, batch["src_txt"], batch["tgt_txt"]):
      predicted.append(ids_to_text(row, tok, sos, eos))
      expected.append(clean_output(tgt_t))
      sources.append(src_t)

    if cfg["validation_size"] and len(predicted) >= cfg["validation_size"]:
      break

  for i in random.sample(range(len(predicted)),
                         min(cfg["num_validation_examples"], len(predicted))):
    print(f"\nSOURCE:    {sources[i]}\n"
          f"TARGET:    {expected[i]}\n"
          f"PREDICTED: {predicted[i]}")

  bleu = torchmetrics.text.BLEUScore()(predicted, [[e] for e in expected]).item()
  print(f"\nVALIDATION (greedy) over {len(predicted)} sentences | BLEU {bleu:.3f}")
  return bleu


def save_checkpoint(cfg, raw_model, opt, sched, epoch, step):
  Path(weights_folder(cfg)).mkdir(parents=True, exist_ok=True)
  fname = get_weights_file_path(cfg, f"{step:08d}")
  torch.save({"epoch": epoch, "step": step, "model": raw_model.state_dict(),
              "optimizer": opt.state_dict(), "scheduler": sched.state_dict()}, fname)
  print(f"Saved {fname}")


def train_model(cfg):
  assert torch.cuda.is_available(), "this codebase is CUDA-only"
  torch.backends.cudnn.benchmark = True
  device = torch.device("cuda")
  torch.set_float32_matmul_precision("high")

  train_dl, val_dl, tok = get_ds(cfg)
  total_steps = len(train_dl)
  cfg["warmup_steps"] = max(1, round(0.1 * total_steps))
  set_seed(cfg["seed"])

  model = Transformer(cfg["d_model"], cfg["h"], cfg["N"],
                      tok.get_vocab_size(), cfg["max_seq"],
                      dropout=cfg["dropout"]).to(device)
  raw_model = model
  print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

  opt = torch.optim.Adam(model.parameters(), lr=1.0, betas=cfg["betas"], eps=cfg["eps"], fused=True)
  d, w, k = cfg["d_model"], cfg["warmup_steps"], cfg["lr_scale"]
  sched = LambdaLR(opt, lambda s: k * d ** -0.5 * min(max(s, 1) ** -0.5, max(s, 1) * w ** -1.5))


  step = 0


  if cfg["use_compile"]:
    model = torch.compile(model, dynamic=True)

  pad_id = tok.token_to_id("[PAD]")
  loss_fn = nn.CrossEntropyLoss(ignore_index=pad_id,
                                label_smoothing=cfg["label_smoothing"])
  eval_loss_fn = nn.CrossEntropyLoss(ignore_index=pad_id, reduction="sum")

  vocab = tok.get_vocab_size()

  for epoch in range(cfg["num_epochs"]):
    model.train()
    opt.zero_grad(set_to_none=True)
    it = tqdm(train_dl, desc=f"Epoch {epoch:02d}")
    ema = None
    seen = 0
    next_val, next_bleu = cfg["val_loss_every_pairs"], cfg["bleu_every_pairs"]
    for batch in it:
      enc = batch["enc_input"].to(device, non_blocking=True)
      dec = batch["dec_input"].to(device, non_blocking=True)
      enc_mask = batch["enc_mask"].to(device, non_blocking=True)
      label = batch["label"].to(device, non_blocking=True)

      with torch.autocast("cuda", dtype=torch.bfloat16):
        logits = model(enc, enc_mask, dec)
        loss = loss_fn(logits.view(-1, vocab), label.view(-1))

      loss.backward()
      opt.step()
      opt.zero_grad(set_to_none=True)
      sched.step()
      step += 1

      cur = loss.item()
      ema = cur if ema is None else 0.98 * ema + 0.02 * cur
      it.set_postfix(avg=f"{ema:6.3f}", lr=f"{opt.param_groups[0]['lr']:.2e}")
      seen += enc.size(0)
      if seen >= next_val or seen >= next_bleu:
        if seen >= next_val:
          next_val += cfg["val_loss_every_pairs"]
          vl = run_validation_loss(raw_model, val_dl, eval_loss_fn, pad_id, vocab, device)
          it.write(f"[{seen:,} pairs] val loss {vl:6.3f} | val ppl {math.exp(vl):8.2f}")
        if seen >= next_bleu:
          next_bleu += cfg["bleu_every_pairs"]
          run_validation(raw_model, val_dl, tok, cfg, device)
          save_checkpoint(cfg, raw_model, opt, sched, epoch, step)
        model.train()
    val_loss = run_validation_loss(raw_model, val_dl, eval_loss_fn, pad_id, vocab, device)
    print(f"Epoch {epoch:02d} | train {ema:6.3f} | val loss {val_loss:6.3f}"
          f"| val ppl {math.exp(val_loss):8.2f}")


    run_validation(raw_model, val_dl, tok, cfg, device)
    model.train()
    save_checkpoint(cfg, raw_model, opt, sched, cfg["num_epochs"] - 1, step)



In [23]:
import torch
from tokenizers import Tokenizer

def load(cfg, epoch=None):
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  tok = Tokenizer.from_file(str(tokenizer_path(cfg, "shared")))
  model = Transformer(cfg["d_model"], cfg["h"], cfg["N"],
                      tok.get_vocab_size(), cfg["max_seq"], cfg["dropout"]).to(device)
  ckpt = get_weights_file_path(cfg, epoch) if epoch else latest_weights_file_path(cfg)
  assert ckpt, "no checkpoint found -- train first"
  print(f"Loading {ckpt}")
  model.load_state_dict(torch.load(ckpt, map_location=device)["model"])
  model.eval()
  return model, tok, device

@torch.no_grad()
def translate(sentence, model, tok, cfg, device):
  sos, eos = tok.token_to_id("[SOS]"), tok.token_to_id("[EOS]")
  ids = tok.encode(sentence).ids[: cfg["max_seq"] - 2]
  src = torch.tensor([[sos, *ids, eos]], device=device)
  src_mask = torch.ones(1, 1, 1, src.size(1), dtype=torch.bool, device=device)
  out = beam_search_decode(model, src, src_mask, sos, eos,
                           cfg["max_seq"], cfg["beam_size"], cfg["length_penalty"])
  out = out.tolist()[1:]
  if eos in out:
    out = out[: out.index(eos)]
  return clean_output(tok.decode(out))

In [24]:
cfg = get_config()
train_model(cfg)
model, tok, device = load(cfg)

dataset: loading /content/drive/MyDrive/mt_cache/ds_1c013c18f2cc.pkl
dataset: loaded in 46s
Vocab 32000 | train 3837976 val 38948
Parameters: 60,487,680


Epoch 00:   3%|▎         | 393/14993 [04:50<2:44:36,  1.48it/s, avg=7.170, lr=3.00e-04]

[100,096 pairs] val loss  6.844 | val ppl   937.98


Epoch 00:   5%|▌         | 785/14993 [05:17<1:17:09,  3.07it/s, avg=6.222, lr=5.99e-04]

[200,192 pairs] val loss  5.852 | val ppl   347.96


...

[3,700,224 pairs] val loss  2.367 | val ppl    10.67


Epoch 00:  99%|█████████▉| 14847/14993 [21:35<00:48,  3.01it/s, avg=3.000, lr=3.63e-04]

[3,800,088 pairs] val loss  2.377 | val ppl    10.77


Epoch 00: 100%|██████████| 14993/14993 [21:44<00:00, 11.49it/s, avg=3.016, lr=3.61e-04]


Epoch 00 | train  3.016 | val loss  2.357| val ppl    10.56

SOURCE:    You will find here a lot of products of Heath Ledger.
TARGET:    Vous trouverez ici de nombreux produits de Heath Ledger aux meilleurs prix.
PREDICTED: Vous trouverez ici de nombreux produits de Heath Ledger aux meilleurs prix.

SOURCE:    Your best word count in one day?? ?
TARGET:    Who did I meet today at the TC in Highlands Ranch?
PREDICTED: Votre meilleur mot compte en un jour??

SOURCE:    Serve the kids with some Mac and Cheese.
TARGET:    Servir les enfants avec des Mac et fromage.
PREDICTED: Servez les enfants avec un Mac et du fromage.

VALIDATION (greedy) over 3072 sentences | BLEU 0.273
Saved wmt_wmt14_weights/tmodel_00014993.pt
Loading wmt_wmt14_weights/tmodel_00014993.pt


In [62]:
import textwrap

text = translate("I'm Abdou, a student at UofT interested in computer science, especially how to build intelligent systems to \
                  accelerate science discoveries and help researchers tackle some of the world's most challenging problems..", model, tok, cfg, device)
print(textwrap.fill(text, 140))

Je suis Abdou, étudiant à UofT intéressé par les sciences informatiques, notamment comment construire des systèmes intelligents pour
accélérer les découvertes scientifiques et aider les chercheurs à s'attaquer à certains des problèmes les plus difficiles du monde.


In [67]:
import textwrap

text = translate("I'm Abdou, a student at UofT with a strong interest in computer science, \
especially AI and its use in science. I'm fascinated by the idea that we can build intelligent systems \
that help scientists analyze data, test ideas, and discover things that are revolutionanry. \
I want to better understand how these systems work and how we can make them more useful and reliable. \
My goal is to develop the skills needed to build AI systems that can help accelerate research and contribute to \
solving important scientific problems.", model, tok, cfg, device)
print(textwrap.fill(text, 140))

Je suis Abdou, étudiant à UofT avec un vif intérêt pour la science informatique, particulièrement AI et son utilisation en sciences. Je suis
fasciné par l'idée que nous pouvons bâtir des systèmes intelligents qui aident les scientifiques à analyser les données, à tester les idées
et à découvrir les choses qui sont révolutionnaires, et je veux mieux comprendre comment ces systèmes fonctionnent et nous pouvons les
rendre plus utiles et fiables. Mon objectif est de développer les compétences nécessaires pour construire des systèmes d'AI qui peuvent
aider à accélérer la recherche et contribuer à résoudre d'importants problèmes scientifiques.


In [69]:
import os, shutil

shutil.copy("wmt_wmt14_weights/tmodel_00014993.pt", "/content/drive/MyDrive/mt_cache/")

'/content/drive/MyDrive/mt_cache/tmodel_00014993.pt'